In [9]:
from pyspark.sql import SparkSession

# Inisialisasi Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Test_Notebook") \
    .getOrCreate()

print("Spark is ready!")
print(f"Spark version: {spark.version}")

Spark is ready!
Spark version: 4.1.1


26/03/09 14:05:35 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
from pyspark.sql import SparkSession
import os

# Inisialisasi Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Homework_Q2") \
    .getOrCreate()

# 1. Read data dari folder raw
# Notebook berada di code/notebook, sehingga path relatif perlu naik ke root project
base_dir = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
input_path = os.path.join(base_dir, "data", "raw", "yellow_tripdata_2025-11.parquet")
df = spark.read.parquet(input_path)

# 2. Repartition menjadi 4 bagian
df_repartitioned = df.repartition(4)

# 3. Tulis ke folder data/pq/
output_path = os.path.join(base_dir, "data", "pq", "yellow_2025_11_repartitioned")
df_repartitioned.write.mode("overwrite").parquet(output_path)

print("Proses Repartition Selesai.")

Proses Repartition Selesai.


In [4]:
from pyspark.sql.functions import col, to_date

trips_on_15th = df.filter(to_date(col("tpep_pickup_datetime")) == "2025-11-15").count()
print(trips_on_15th)

162604


In [6]:
from pyspark.sql import functions as F

# Menambahkan kolom duration_hours
df_duration = df.withColumn(
    "duration_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

# Mencari nilai maksimum
max_duration = df_duration.select(F.max("duration_hours")).collect()[0][0]

print(f"The longest trip duration is: {max_duration} hours")

The longest trip duration is: 90.64666666666666 hours


In [5]:
from pyspark.sql import SparkSession
import os

# pastikan SparkSession tersedia (misal jika kernel di-restart)
if "spark" not in globals():
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("Homework_Q2") \
        .getOrCreate()

# pastikan base_dir tersedia (misal jika kernel di-restart)
if "base_dir" not in globals():
    base_dir = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

# pastikan df tersedia (misal jika kernel di-restart)
if "df" not in globals():
    input_path = os.path.join(base_dir, "data", "raw", "yellow_tripdata_2025-11.parquet")
    df = spark.read.parquet(input_path)

# 1. Membaca data lookup zona dari folder data/raw
# Notebook berada di code/notebook, sehingga path relatif perlu naik ke root project
zones_path = os.path.join(base_dir, "data", "raw", "taxi_zone_lookup.csv")
zones = spark.read.option("header", True).csv(zones_path)

# 2. Mendaftarkan DataFrame sebagai temporary view
zones.createOrReplaceTempView("zones")
df.createOrReplaceTempView("trips")

# 3. Menjalankan Spark SQL untuk mencari zona dengan jemputan paling sedikit
spark.sql("""
SELECT 
    z.Zone, 
    COUNT(*) as trips_count
FROM 
    trips t
JOIN 
    zones z ON t.PULocationID = z.LocationID
GROUP BY 
    z.Zone
ORDER BY 
    trips_count ASC
LIMIT 1
""").show()

+--------------------+-----------+
|                Zone|trips_count|
+--------------------+-----------+
|Governor's Island...|          1|
+--------------------+-----------+

